# LangGraph + LangChain + LangSmith on AgentCore (TravelMind)

**Track:** Agentic AI Bootcamp &nbsp;|&nbsp; **Level:** Advanced

Read `07_langchain_langgraph_langsmith_with_agentcore.md` first. Here we build the three **combine** patterns in code:
- LangGraph checkpointer **backed by AgentCore Memory** (`AgentCoreMemorySaver`) for durable state
- LangGraph store **backed by AgentCore Memory** (`AgentCoreMemoryStore`) for long-term memory (pattern)
- **LangSmith** tracing on, alongside AgentCore hosting + Observability

Then wrap the graph in a Runtime entrypoint and deploy.

Job map: **LangChain = models/prompts, LangGraph = orchestration + state, LangSmith = trace/eval, AgentCore = operations.**


## Four tools, four jobs

```mermaid
flowchart TD
    subgraph Logic[Agent logic]
        LC[LangChain: model, prompts]
        LG[LangGraph: graph, state, checkpoint]
        LC --> LG
    end
    subgraph Ops[AgentCore: operations]
        RT[Runtime: host]
        MEM[Memory: durable backend]
    end
    LS[LangSmith: trace + eval]
    Logic -->|runs on| Ops
    Logic -.traces to.-> LS
```


## 0. Setup

**VS Code:** venv as kernel, `aws configure` (`us-east-1`), install deps below.
**Colab:** same `pip install` first; credentials via secrets/env.
**Model:** `us.anthropic.claude-haiku-4-5-20251001-v1:0` enabled in Bedrock.


In [ ]:
%pip install -q --upgrade "boto3>=1.39.9" langchain langgraph langgraph-checkpoint-aws langchain-aws bedrock-agentcore

## 1. The graph: LangChain model + LangGraph orchestration

`init_chat_model(..., model_provider="bedrock_converse")` is the LangChain model abstraction. `create_react_agent` is the LangGraph orchestration. A tool gives the loop something to call.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# LangChain: model abstraction over Bedrock
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse")

# a tool (in-process; in prod this comes from Gateway)
@tool
def get_pnr(pnr: str) -> str:
    """Look up a booking by PNR."""
    db = {"JX48Q2": {"passenger": "Rao", "tier": "Gold",
                     "segment": "BLR-DEL", "status": "CANCELLED"}}
    return json.dumps(db.get(pnr, {"error": "not found"}))

print("model + tool ready")


## 2. Combine #1: LangGraph checkpointer, AgentCore Memory backend

LangGraph gives the checkpointer *interface*; AgentCore Memory is the durable *backend*. You write normal LangGraph and state survives restarts and scales across isolated VMs. Route state with `actor_id` + `thread_id`.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

memc = MemoryClient(region_name=REGION)
stm = memc.create_memory_and_wait(
    name=f"tm-lg-stm-{uuid.uuid4().hex[:8]}", strategies=[], event_expiry_days=7)
MEM_ID = stm["id"]
print("memory:", MEM_ID)


In [ ]:
from langgraph_checkpoint_aws import AgentCoreMemorySaver

checkpointer = AgentCoreMemorySaver(MEM_ID, region_name=REGION)

# LangGraph agent, compiled with the AgentCore-backed checkpointer
agent = create_react_agent(
    llm, tools=[get_pnr], checkpointer=checkpointer,
    prompt="You are TravelMind, an airline support agent. Be concise and accurate.",
)

# actor_id + thread_id route the persisted state
THREAD = f"pnr-{PNR}-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"actor_id": PASSENGER, "thread_id": THREAD}}

r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "PNR JX48Q2 BLR-DEL was cancelled. Status?"}]},
    config)
print(r1["messages"][-1].content)


In [ ]:
# same thread -> the checkpointer restored state, so it remembers the PNR
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Which segment was cancelled again?"}]},
    config)
print(r2["messages"][-1].content)


## 3. Combine #2: LangGraph store, AgentCore Memory backend (long-term)

`AgentCoreMemoryStore` puts LangGraph's store interface on top of AgentCore Memory's async extraction. Save in a pre-model hook, search to inject context. Shown as a pattern (extraction is eventually consistent, ~1 min).

In [ ]:
from langgraph_checkpoint_aws import AgentCoreMemoryStore

store = AgentCoreMemoryStore(MEM_ID, region_name=REGION)

def pre_model_hook(state, config, *, store):
    actor_id  = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    msgs = state.get("messages", [])
    # save the latest human message for background extraction
    for m in reversed(msgs):
        role = getattr(m, "type", None) or (m.get("role") if isinstance(m, dict) else None)
        if role in ("human", "user"):
            store.put((actor_id, thread_id), str(uuid.uuid4()), {"message": str(m)})
            break
    # retrieve long-term memories to prepend (preferences, facts)
    try:
        hits = store.search(("preferences", actor_id), query="traveler preferences", limit=5)
        # append 'hits' into the model input per your context strategy
    except Exception as e:
        print("store.search note:", e)
    return {"model_input_messages": state["messages"]}

# wire into a react agent:
# agent_lt = create_react_agent(llm, tools=[get_pnr], store=store,
#                               pre_model_hook=pre_model_hook, checkpointer=checkpointer)
print("long-term store pattern defined")


## 4. Combine #3: LangSmith tracing + AgentCore hosting

LangSmith traces the graph internals (nodes, LLM calls, tokens) for debugging and eval. AgentCore Observability handles session/ops metrics. Run both.

In [ ]:
# Turn on LangSmith tracing (inject the key via config/secrets in real deployments)
os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "<your-key>"
os.environ["LANGSMITH_PROJECT"] = "travelmind-agentcore"

# With the key set, agent.invoke(...) traces appear in your LangSmith project.
# AgentCore Observability (session metrics) is separate and turns on when hosted on Runtime.
print("LangSmith tracing configured:", os.environ.get("LANGSMITH_TRACING"))
print("Set LANGSMITH_API_KEY to send traces. Coexists with AgentCore Observability.")


**LangSmith vs AgentCore Observability, side by side**

| Task | Use |
|---|---|
| "Why did the graph take this branch?" | LangSmith |
| "Run this agent over a labeled eval dataset" | LangSmith |
| "P99 session latency + error rate of the deployed service" | AgentCore Observability |
| "One pane for everything" | Export AgentCore OTEL to LangSmith |

## 5. Wrap for Runtime and deploy

The graph, hosted. `context.session_id` becomes the LangGraph `thread_id`; the checkpointer keeps state durable.

In [ ]:
deploy_file = '''
import os, json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph_checkpoint_aws import AgentCoreMemorySaver

os.environ.setdefault("LANGSMITH_TRACING", "true")   # LangSmith key injected via config

REGION   = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MEM_ID   = os.environ["MEMORY_ID"]                    # inject, do not hardcode

# build time
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse")

@tool
def get_pnr(pnr: str) -> str:
    db = {"JX48Q2": {"passenger":"Rao","tier":"Gold","segment":"BLR-DEL","status":"CANCELLED"}}
    return json.dumps(db.get(pnr, {"error":"not found"}))

agent = create_react_agent(
    llm, tools=[get_pnr],
    checkpointer=AgentCoreMemorySaver(MEM_ID, region_name=REGION),
    prompt="You are TravelMind. Be concise and accurate.")

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload, context):
    # context.session_id (>=16 chars) is the LangGraph thread_id
    result = agent.invoke(
        {"messages": [{"role": "user", "content": payload["prompt"]}]},
        config={"configurable": {"actor_id": payload.get("actor_id", "Rao"),
                                 "thread_id": context.session_id}})
    return {"result": result["messages"][-1].content}

if __name__ == "__main__":
    app.run()
'''
with open("langgraph_runtime.py", "w") as f:
    f.write(deploy_file)
print("wrote langgraph_runtime.py")


**Deploy (terminal):**

```bash
# new CLI
npm install -g @aws/agentcore
agentcore create        # framework LangGraph
agentcore deploy
agentcore invoke --prompt "PNR JX48Q2 refund?" --session-id "$(uuidgen)"

# legacy toolkit
pip install bedrock-agentcore-starter-toolkit
agentcore configure -e langgraph_runtime.py
agentcore launch
```

**Production notes (post-cell):**
- `MEMORY_ID`, `LANGSMITH_API_KEY`, region: inject via env/config, never hardcode.
- IAM roles, not access keys. IAM action is `bedrock:InvokeModel`, not `bedrock:Converse`.
- In-memory checkpointer is a production anti-pattern; the `AgentCoreMemorySaver` above is the fix.
- Shared/authed tools -> Gateway + Identity; add retries/timeouts around model + tool calls.


---
## 6. Cleanup

In [ ]:
if globals().get("MEM_ID"):
    try: memc.delete_memory(memory_id=MEM_ID); print("deleted memory", MEM_ID)
    except Exception as e: print("mem:", e)
# DeleteAgentRuntime for any runtime you deployed.


---
## The four-tool summary (say the owners instantly)

| Concern | Owner |
|---|---|
| Orchestration (graph, branching, cycles) | LangGraph |
| Model / prompt abstraction | LangChain |
| Dev tracing + eval | LangSmith |
| Durable state / long-term memory backend | AgentCore Memory |
| Ops / session metrics | AgentCore Observability |
| Tools at scale + auth | AgentCore Gateway + Identity |
| Hosting | AgentCore Runtime |

If you can name the owner for each without hesitating, you have the production model down.

**End of series.** Foundations, features, harness (concept + by-hand + managed), Strands production, and the LangChain/LangGraph/LangSmith stack, each with the feature-ownership decisions that keep the architecture honest.